In [27]:
import os
import pandas as pd

# ==============================================================================
# 1. CONFIGURACIÓ I RUTES
# ==============================================================================
# Utilitzem './' per assegurar que troba la ruta des de la carpeta actual
carpeta_entrada = '../data/processed/netejats/'
ruta_sortida = '../data/processed/03_master_dataset_final.csv'

fitxers = [f for f in os.listdir(carpeta_entrada) if f.endswith('.csv')]
print(f"🔗 Iniciant el Mega-Merge amb {len(fitxers)} fitxers...\n")

columnes_brossa = ['any', 'codi_districte', 'nom_districte', 'codi_barri', 
                   'nom_barri', 'seccio_censal', 'codi_sc', 'seccio', 'districte', 'aeb', 'data_referencia']

# ==============================================================================
# 2. CREACIÓ DE L'ESQUELET GEOGRÀFIC
# ==============================================================================
print("🏗️ Construint l'esquelet base de les Seccions Censals...")
dades_geo = []

for arxiu in fitxers:
    df = pd.read_csv(os.path.join(carpeta_entrada, arxiu), low_memory=False)
    cols_actuals = [c.lower() for c in df.columns]
    
    if 'nom_districte' in cols_actuals and 'nom_barri' in cols_actuals:
        nom_dist_col = df.columns[cols_actuals.index('nom_districte')]
        nom_barri_col = df.columns[cols_actuals.index('nom_barri')]
        
        geo_df = df[['CODI_UNIC', nom_dist_col, nom_barri_col]].copy()
        geo_df.columns = ['CODI_UNIC', 'Nom_Districte', 'Nom_Barri']
        dades_geo.append(geo_df)

df_master = pd.concat(dades_geo).drop_duplicates(subset=['CODI_UNIC']).reset_index(drop=True)
df_master = df_master.sort_values(by='CODI_UNIC').reset_index(drop=True)
print(f"✅ Esquelet creat amb {len(df_master)} seccions úniques.\n")

# ==============================================================================
# 3. EL BUCLE DE FUSIÓ
# ==============================================================================
for arxiu in fitxers:
    df = pd.read_csv(os.path.join(carpeta_entrada, arxiu), low_memory=False)
    
    parts_nom = arxiu.replace('.csv', '').split('_')
    prefix = "_".join(parts_nom[1:3]) if len(parts_nom) > 2 else parts_nom[0]
    
    columnes_a_guardar = ['CODI_UNIC']
    for col in df.columns:
        if col == 'CODI_UNIC': continue
        if col.lower() not in columnes_brossa:
            columnes_a_guardar.append(col)
            
    df_net = df[columnes_a_guardar].copy()
    
    noves_columnes = {col: f"{prefix}_{col}" for col in df_net.columns if col != 'CODI_UNIC'}
    df_net.rename(columns=noves_columnes, inplace=True)
    
    df_net = df_net.groupby('CODI_UNIC').mean(numeric_only=True).reset_index()
    
    df_master = pd.merge(df_master, df_net, on='CODI_UNIC', how='left')
    print(f"   ➕ Enganxades variables de: {arxiu}")

# ==============================================================================
# 4. IMPUTACIÓ ESPACIAL INTEL·LIGENT (LA MÀGIA)
# ==============================================================================
print("\n🗺️ Escanejant valors buits per aplicar Imputació Espacial...")

# Busquem automàticament quines columnes numèriques tenen valors buits (NaN)
cols_a_ignorar = ['CODI_UNIC', 'Nom_Districte', 'Nom_Barri']
columnes_amb_nuls = [col for col in df_master.columns if col not in cols_a_ignorar and df_master[col].isnull().any()]

for col in columnes_amb_nuls:
    # Només ho fem si la columna és de números (no volem fer mitjanes de text)
    if pd.api.types.is_numeric_dtype(df_master[col]):
        nuls_abans = df_master[col].isnull().sum()
        
        # 1. Creem el "Xivat" (Flag) amb un 1 on hi havia un buit i un 0 on hi havia dada real
        nom_flag = f"{col}_Imputat"
        df_master[nom_flag] = df_master[col].isnull().astype(int)
        
        # 2. Imputem la mitjana del BARRI
        mitjana_barri = df_master.groupby('Nom_Barri')[col].transform('mean')
        df_master[col] = df_master[col].fillna(mitjana_barri)
        
        # 3. PLA DE SEGURETAT: Si el Barri sencer estava buit, usem el DISTRICTE
        mitjana_districte = df_master.groupby('Nom_Districte')[col].transform('mean')
        df_master[col] = df_master[col].fillna(mitjana_districte)
        
        nuls_ara = df_master[col].isnull().sum()
        reparats = nuls_abans - nuls_ara
        
        print(f"   🪄 {col}: {reparats} forats omplerts. (Columna '{nom_flag}' afegida al dataset)")

# ==============================================================================
# 5. EXPORTACIÓ I AUDITORIA
# ==============================================================================
print("\n🧹 Netejant decimals infinits abans de guardar...")
# Arrodonim totes les columnes numèriques a 2 decimals
df_master = df_master.round(1)

df_master.to_csv(ruta_sortida, index=False)
print(f"\n🎉 MEGA-MERGE COMPLETAT!")
print(f"📊 Dimensions finals del Dataset: {df_master.shape[0]} files x {df_master.shape[1]} columnes.")
print(f"💾 Arxiu desat a: {ruta_sortida}")

# Comprovació final de qualitat
print("\n--- RESUM DE QUALITAT FINAL (Han de quedar 0 forats a les dades numèriques) ---")
nuls_totals = df_master.isnull().sum()
nuls_importants = nuls_totals[nuls_totals > 0].sort_values(ascending=False).head(5)

if len(nuls_importants) > 0:
    print("⚠️ Encara queden algunes dades buides:")
    print(nuls_importants)
else:
    print("🌟 ESPECTACULAR: Dataset 100% ple i preparat per a la Xarxa Neuronal!")

🔗 Iniciant el Mega-Merge amb 7 fitxers...

🏗️ Construint l'esquelet base de les Seccions Censals...
✅ Esquelet creat amb 1068 seccions úniques.

   ➕ Enganxades variables de: 2022_renda_disponible_llars_per_persona.csv
   ➕ Enganxades variables de: 2023_atles_renda_bruta_llar.csv
   ➕ Enganxades variables de: 2023_atles_renda_mitjana.csv
   ➕ Enganxades variables de: 2025_pad_dom_mdbas_edat-6599.csv
   ➕ Enganxades variables de: 2026_edificacions_any_con.csv
   ➕ Enganxades variables de: 2026_edificacions_edat_mitjana.csv
   ➕ Enganxades variables de: 2026_edificacions_superficie.csv

🗺️ Escanejant valors buits per aplicar Imputació Espacial...
   🪄 edificacions_edat_Edat_mitjana: 13 forats omplerts. (Columna 'edificacions_edat_Edat_mitjana_Imputat' afegida al dataset)

🧹 Netejant decimals infinits abans de guardar...

🎉 MEGA-MERGE COMPLETAT!
📊 Dimensions finals del Dataset: 1068 files x 12 columnes.
💾 Arxiu desat a: ../data/processed/03_master_dataset_final.csv

--- RESUM DE QUALITAT 